# CLM-v0.1-8B en Google Colab

[Contrastive Language Model (CLM)](https://huggingface.co/Contrastive-LM/CLM-v0.1-8B): un modelo **decisional** (System One).
No genera texto: dado un *estado* y un conjunto cerrado de *candidatos*, devuelve una distribución de probabilidad sobre ellos.

Arquitectura: **Qwen3-8B congelado** (embedding del último token, servido con vLLM) + dos cabezas MLP (~20M parámetros, 75 MB) entrenadas con InfoNCE.

**Requisitos:** Colab Pro con GPU **L4 (24 GB) o A100**. Una T4 (16 GB) no tiene memoria suficiente para Qwen3-8B en bf16.
`Entorno de ejecución → Cambiar tipo de entorno → L4 / A100`.

Contenido:
1. Instalación y arranque del encoder (vLLM)
2. Preguntas tipadas (`noul` / `choice` / `score`)
3. Ranking de candidatos libres (selección de herramientas, best-of-N)
4. Mini-benchmark zero-shot en AG News: CLM vs. ablación sin cabezas (`clm-raw`) vs. BART-MNLI
5. Latencia y efecto de la caché de acciones
6. (Opcional) Playground web

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Instalación

`contrastive-lm` instala también `vllm` (y su propia versión de `torch`). Si tras instalar aparece un error de import,
haz **Entorno de ejecución → Reiniciar sesión** y continúa desde la celda siguiente (no hace falta reinstalar).

In [ ]:
!pip install -q contrastive-lm datasets

In [ ]:
# Arranca el encoder Qwen3-8B (pooling, último token) en segundo plano.
# Se deja ~20% de VRAM libre para BART-MNLI en el benchmark.
import subprocess, time, requests

vllm_log = open("vllm.log", "w")
vllm_proc = subprocess.Popen(
    ["vllm", "serve", "Qwen/Qwen3-8B",
     "--served-model-name", "qwen3-8b",
     "--runner", "pooling",
     "--max-model-len", "2048",
     "--enable-prefix-caching",
     "--gpu-memory-utilization", "0.80",
     "--port", "8090"],
    stdout=vllm_log, stderr=subprocess.STDOUT)

# La primera vez descarga ~16 GB de pesos: puede tardar varios minutos.
t0 = time.time()
while True:
    try:
        if requests.get("http://127.0.0.1:8090/v1/models", timeout=2).status_code == 200:
            break
    except requests.RequestException:
        pass
    if vllm_proc.poll() is not None:
        raise RuntimeError("vLLM ha terminado; revisa vllm.log:\n" + open("vllm.log").read()[-3000:])
    time.sleep(10)
print(f"Encoder listo en {time.time() - t0:.0f} s")

In [ ]:
# Motor de CLM en el propio proceso (sin servidor HTTP).
from clm import Engine
from clm.heads import download

ckpt = download()   # descarga CLM_v0.1-8B.pt (75 MB) a ~/.cache/clm/
engine = Engine(checkpoint=ckpt, emb_url="http://127.0.0.1:8090/v1/embeddings", device="cuda")
print(engine.models())

## 2. Preguntas tipadas

- `noul`: sí/no → probabilidad de que la afirmación sea cierta.
- `choice`: elegir una opción entre varias (cada una con su descripción).
- `score`: escala ordenada → nivel esperado.

Las probabilidades son **relativas al conjunto de candidatos**: CLM no puede decir "ninguna de las anteriores" si no se la ofreces.

In [ ]:
import json

state = "Customer: my invoice was charged twice and nobody answers the phone!"
questions = {
    "urgency": {"type": "noul", "instructions": "Is this urgent?"},
    "department": {"type": "choice", "instructions": "Which team should handle this?",
                   "criteria": {"billing": "Charges, invoices, refunds",
                                "technical": "Bugs and outages",
                                "sales": "New plans, pricing questions, upgrades"}},
    "frustration": {"type": "score", "instructions": "How frustrated is the customer?",
                    "criteria": ["Calm", "Frustrated", "Very angry"]},
}
out = engine.answer(state, questions)
print(json.dumps(out, indent=2))

In [ ]:
# Mismo caso en español: el modelo se entrenó en inglés, así que conviene comprobar cómo se degrada.
state_es = "Cliente: ¡me habéis cobrado la factura dos veces y nadie coge el teléfono!"
questions_es = {
    "department": {"type": "choice", "instructions": "¿Qué equipo debe encargarse?",
                   "criteria": {"billing": "Cobros, facturas, reembolsos",
                                "technical": "Errores y caídas del servicio",
                                "sales": "Planes nuevos, precios, mejoras de plan"}},
}
print(engine.answer(state_es, questions_es)["answers"]["department"])

## 3. Ranking de candidatos libres

`rank` es la primitiva sobre la que se construye todo: puntuar candidatos contra un estado.
Sirve para elegir herramientas, escoger la mejor de N respuestas o decidir la siguiente acción de un agente.

In [ ]:
from pprint import pprint

pprint(engine.rank("What causes tides on Earth?",
                   ["The Moon's gravitational pull.",
                    "Photosynthesis in plants.",
                    "Because the Earth is round."]))

# Selección de herramienta
tools = {
    "get_weather": "Get the current weather forecast for a city",
    "send_email": "Send an email to a recipient with a subject and body",
    "search_flights": "Search flights between two airports on a date",
    "create_calendar_event": "Create a meeting in the user's calendar",
}
pprint(engine.rank("User: book me something to fly from Madrid to Paris next Friday",
                   list(tools.values()), instructions="Which tool should the agent call next?"))

# Best-of-N: ¿qué solución es correcta?
pprint(engine.rank("Write a Python function that returns the n-th Fibonacci number (fib(0)=0).",
                   ["def fib(n):\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a + b\n    return a",
                    "def fib(n):\n    return fib(n-1) + fib(n-2)",
                    "def fib(n):\n    return n * (n + 1) // 2"],
                   instructions="Which candidate solution is correct?"))

## 4. Mini-benchmark zero-shot: AG News

Clasificación de noticias en 4 clases sin entrenar nada. Comparamos:

- **`clm-latest`**: CLM con sus cabezas entrenadas.
- **`clm-raw`**: misma Qwen3-8B, coseno directo sin cabezas (ablación: mide cuánto aportan las cabezas).
- **BART-large-MNLI**: el clasificador zero-shot clásico de Hugging Face (`zero-shot-classification`).

Es una prueba pequeña e indicativa, no un benchmark oficial.

In [ ]:
from datasets import load_dataset

N = 400
ds = load_dataset("fancyzhx/ag_news", split="test").shuffle(seed=0).select(range(N))
labels = ["World", "Sports", "Business", "Sci/Tech"]
criteria = {
    "World": "World news: international politics, conflicts, governments, diplomacy",
    "Sports": "Sports news: games, teams, athletes, tournaments",
    "Business": "Business news: companies, markets, economy, finance",
    "Sci/Tech": "Science and technology news: computers, internet, software, space, research",
}
question = {"topic": {"type": "choice", "instructions": "What is the topic of this news article?",
                      "criteria": criteria}}

In [ ]:
import time
import numpy as np

def eval_clm(model):
    preds, lat = [], []
    for ex in ds:
        t = time.perf_counter()
        a = engine.answer(ex["text"], question, model=model)["answers"]["topic"]
        lat.append(time.perf_counter() - t)
        preds.append(labels.index(a["choice"]))
    acc = float(np.mean(np.array(preds) == np.array(ds["label"])))
    return acc, 1000 * float(np.median(lat)), preds

results = {}
for m in ["clm-latest", "clm-raw"]:
    acc, ms, _ = eval_clm(m)
    results[m] = (acc, ms)
    print(f"{m:12s} accuracy={acc:.3f}  latencia mediana={ms:.1f} ms/ejemplo")

In [ ]:
import torch
from transformers import pipeline

zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli",
               device=0, torch_dtype=torch.float16)
cand = list(criteria.values())
preds, lat = [], []
for ex in ds:
    t = time.perf_counter()
    r = zsc(ex["text"], candidate_labels=cand)
    lat.append(time.perf_counter() - t)
    preds.append(cand.index(r["labels"][0]))
acc = float(np.mean(np.array(preds) == np.array(ds["label"])))
results["bart-large-mnli"] = (acc, 1000 * float(np.median(lat)))
print(f"bart-large-mnli accuracy={acc:.3f}  latencia mediana={results['bart-large-mnli'][1]:.1f} ms/ejemplo")

In [ ]:
import pandas as pd
pd.DataFrame(results, index=["accuracy", "ms/ejemplo (mediana)"]).T.round(3)

## 5. Caché de acciones

Los estados y las acciones se codifican por separado: los embeddings de los candidatos se calculan una sola vez
y se reutilizan. Con muchos candidatos (p. ej. 1.000 herramientas o documentos), el coste de una decisión nueva
es básicamente **un embedding del estado + un producto escalar**.

In [ ]:
cands = [f"Option number {i}: {w}" for i, w in enumerate(
    ["open the door", "close the window", "turn left", "turn right", "jump", "wait",
     "pick up the key", "drop the key", "talk to the guard", "read the sign"] * 100)]

t = time.perf_counter(); engine.rank("You are in front of a locked door. You have a key.", cands)
cold = time.perf_counter() - t
t = time.perf_counter(); engine.rank("You see a guard standing next to a sign.", cands)
warm = time.perf_counter() - t
print(f"{len(cands)} candidatos -> caché fría: {cold*1000:.0f} ms | caché caliente (estado nuevo): {warm*1000:.0f} ms")

## 6. (Opcional) Playground web

`clm-serve` levanta la API HTTP (`/v1/systemone`, `/v1/rank`) y una interfaz web para probar preguntas tipadas.

In [ ]:
import os, subprocess
clm_log = open("clm.log", "w")
clm_proc = subprocess.Popen(["clm-serve", "--port", "8700", "--ckpt", ckpt],
                            stdout=clm_log, stderr=subprocess.STDOUT)
time.sleep(15)

from google.colab import output
output.serve_kernel_port_as_window(8700)

In [ ]:
# Cliente HTTP (misma API que TypeSafe)
from clm import CLMClient, Choice, Noul, Score
client = CLMClient("http://127.0.0.1:8700")
r = client.system_one(state=state, questions={
    "urgency": Noul(instructions="Is this urgent?"),
    "frustration": Score(instructions="How frustrated is the customer?",
                         criteria=["Calm", "Frustrated", "Very angry"]),
})
print(r.answers["urgency"].noul, r.answers["frustration"].score, r.latency_ms)

In [ ]:
# Liberar la GPU al terminar
for p in ("clm_proc", "vllm_proc"):
    if p in globals():
        globals()[p].terminate()